# Примеры к Лекции 4: Протоколы межагентной коммуникации — MCP и A2A

Этот ноутбук демонстрирует реальные MCP-серверы и A2A-сервер — никаких моков.

1. **MCP Multi-Server**: два MCP-сервера (DuckDuckGo + SQLite) через `MultiServerMCPClient`
2. **Матрёшка**: MCP-сервер с LangGraph-графом внутри
3. **A2A Agent Card**: получение Agent Card по HTTP
4. **A2A Task Lifecycle**: JSON-RPC lifecycle (submitted → working → completed)
5. **Комбинация**: shared state + MCP + A2A в одном графе

### Перед запуском — поднимите серверы в отдельных терминалах

Все серверы (MCP и A2A) работают по HTTP. Запустите каждый в своём терминале:

```bash
# Терминал 1 — MCP: поиск (DuckDuckGo)
python mcp_search_server.py          # порт 8001

# Терминал 2 — MCP: база данных (SQLite)
python mcp_db_server.py              # порт 8002

# Терминал 3 — MCP: матрёшка (LangGraph внутри)
python mcp_matryoshka_server.py      # порт 8003

# Терминал 4 — A2A: юридический рецензент
python a2a_server.py                 # порт 5002
```

In [1]:
import json
import sys
from pathlib import Path
from typing import TypedDict

import httpx
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph

sys.path.insert(0, str(Path(".").resolve().parent))
from llm_config import check_api_key, get_llm

assert check_api_key(), "API key not set"

print("Imports OK")

Imports OK


In [2]:
# доп функция
def mcp_text(result) -> str:
    """Извлечь текст из результата MCP-инструмента.

    MCP-инструменты возвращают не строку, а list[dict]:
      [{"type": "text", "text": "содержимое"}, ...]
    Эта функция извлекает текст из такого формата.
    """
    if isinstance(result, list):
        return "\n".join(
            item.get("text", str(item)) if isinstance(item, dict) else str(item)
            for item in result
        )
    return str(result)

In [3]:
# Создаём/обновляем SQLite базу с тестовыми данными
from seed_db import seed

seed()

Database seeded: /Users/joeblack/prog/ai/project-MAS/module_06_multiagent/examples_04_protocols/analytics.db


## Часть 1: MCP Multi-Server с изоляцией инструментов

Два MCP-сервера (уже запущены в терминалах):
- **Search Server** (порт 8001) — поиск через DuckDuckGo (`search_web`, `search_news`)
- **Database Server** (порт 8002) — SQLite с данными о фреймворках (`query_db`, `list_tables`, `table_schema`)

`MultiServerMCPClient` подключается к ним по SSE (Server-Sent Events over HTTP).

> **Примечание:** начиная с `langchain-mcp-adapters` 0.1.0, `MultiServerMCPClient` — **не** async context manager.
> Используем напрямую: `client = MultiServerMCPClient({...})`, затем `await client.get_tools()`.

In [4]:
# Подключаемся к двум MCP-серверам по SSE (они должны быть уже запущены в терминалах)
# uv run mcp_search_server.py
# uv run mcp_db_server.py
client = MultiServerMCPClient(
    {
        "search": {"transport": "sse", "url": "http://localhost:8001/sse"},
        "database": {"transport": "sse", "url": "http://localhost:8002/sse"},
    }
)
all_tools = await client.get_tools()
tools_by_name = {t.name: t for t in all_tools}

print(f"Всего инструментов: {len(all_tools)}")
for t in all_tools:
    print(f"  {t.name}: {t.description[:35]}...")

Всего инструментов: 5
  search_web: Поиск информации в интернете через ...
  search_news: Поиск свежих новостей через DuckDuc...
  query_db: Выполнить SQL-запрос к аналитическо...
  list_tables: Показать все таблицы в базе данных....
  table_schema: Получить схему таблицы (колонки и т...


In [19]:
# Вызываем поиск через DuckDuckGo
print("=== search_web (DuckDuckGo) ===")
search_result = await tools_by_name["search_web"].ainvoke(
    {"query": "MCP protocol AI agents 2026"}
)
print(mcp_text(search_result)[:400])

=== search_web (DuckDuckGo) ===
- Model ContextProtocol- Wikipedia: Agent2Agent– Openprotocolfor communication betweenAIagents.AIgovernance – Guidelines and laws to regulate AIPages displaying short descriptions of redirect targets.
- Революция ИИ в2026году: ключевые тренды, которые... / Хабр: ПримитеMCPкак стандарт интеграцииAI-инструментов. Создавайте внутренние платформы, встраивающие ИИ в опыт разработчика. Определите чёткие


In [20]:
# Вызываем SQLite запрос
print("=== query_db (SQLite) ===")
db_result = await tools_by_name["query_db"].ainvoke(
    {
        "sql": "SELECT name, github_stars, monthly_downloads FROM framework_metrics ORDER BY github_stars DESC"
    }
)
print(mcp_text(db_result))

=== query_db (SQLite) ===
name | github_stars | monthly_downloads
---------------------------------------
CrewAI | 45000 | 3800000
Mastra | 22000 | 1300000
LangGraph | 18000 | 5200000
OpenAI Agents SDK | 15000 | 2100000
Letta | 14000 | 300000
Pydantic AI | 12000 | 1500000
Google ADK | 8000 | 900000
smolagents | 6000 | 400000


In [ ]:
# MAS с изолированными инструментами — каждый агент видит только свои


class ResearchState(TypedDict):
    query: str
    web_facts: str
    db_analysis: str
    report: str


llm = get_llm()

# Исследователь: ТОЛЬКО поисковые инструменты
search_tools = [tools_by_name["search_web"], tools_by_name["search_news"]]
# Аналитик: ТОЛЬКО инструменты базы данных
db_tools = [
    tools_by_name["query_db"],
    tools_by_name["list_tables"],
    tools_by_name["table_schema"],
]

# Исследователь — bind_tools: LLM сам решает, какой инструмент вызвать
llm_with_search = llm.bind_tools(search_tools)
search_tools_by_name = {t.name: t for t in search_tools}


async def researcher_node(state: ResearchState) -> dict:
    # LLM выбирает инструмент и аргументы
    response = llm_with_search.invoke(
        [
            SystemMessage(
                content="Ты — исследователь. Используй доступные инструменты для поиска информации."
            ),
            HumanMessage(content=f"Найди ключевые факты по теме: {state['query']}"),
        ]
    )

    # Выполняем вызовы инструментов, которые выбрал LLM
    results = []
    for tool_call in response.tool_calls:
        tool = search_tools_by_name[tool_call["name"]]
        result = mcp_text(await tool.ainvoke(tool_call["args"]))
        results.append(f"[{tool_call['name']}]: {result}")
        print(f"  [Исследователь] LLM вызвал {tool_call['name']}({tool_call['args']})")

    # LLM обобщает результаты
    summary = llm.invoke(
        [
            SystemMessage(content="Систематизируй найденные факты кратко."),
            HumanMessage(content="\n\n".join(results)),
        ]
    )
    return {"web_facts": summary.content}


async def analyst_node(state: ResearchState) -> dict:
    # Аналитик — прямой вызов: код сам решает, какой запрос выполнить
    db_result = mcp_text(
        await db_tools[0].ainvoke(
            {
                "sql": "SELECT name, github_stars FROM framework_metrics ORDER BY github_stars DESC LIMIT 5"
            }
        )
    )
    response = llm.invoke(
        [
            SystemMessage(
                content="Ты — аналитик. Интерпретируй данные из веба и базы кратко."
            ),
            HumanMessage(
                content=f"Факты:\n{state['web_facts']}\n\nДанные из БД:\n{db_result}"
            ),
        ]
    )
    print("  [Аналитик] Прямой вызов query_db (без bind_tools)")
    return {"db_analysis": response.content}


def writer_node(state: ResearchState) -> dict:
    response = llm.invoke(
        [
            SystemMessage(
                content="Напиши краткий отчёт (3-5 предложений) на основе фактов и анализа."
            ),
            HumanMessage(
                content=f"Факты:\n{state['web_facts']}\n\nАнализ:\n{state['db_analysis']}"
            ),
        ]
    )
    return {"report": response.content}


graph = StateGraph(ResearchState)
graph.add_node("researcher", researcher_node)
graph.add_node("analyst", analyst_node)
graph.add_node("writer", writer_node)
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", END)
app = graph.compile()

print("MAS: исследователь (bind_tools) → аналитик (прямой вызов) → писатель")
result = await app.ainvoke(
    {
        "query": "AI agent frameworks 2026",
        "web_facts": "",
        "db_analysis": "",
        "report": "",
    }
)
print(f"\nОтчёт:\n{result['report']}")

MAS: исследователь (bind_tools) → аналитик (прямой вызов) → писатель
  [Исследователь] LLM вызвал search_web({'query': 'AI agent frameworks 2026 key facts overview'})
  [Исследователь] LLM вызвал search_news({'query': 'AI agent frameworks 2026 news'})
  [Исследователь] LLM вызвал search_web({'query': 'best AI agent frameworks 2025 2026 LangGraph AutoGen CrewAI Semantic Kernel LlamaIndex'})
  [Исследователь] LLM вызвал search_web({'query': 'AI agent frameworks market trends 2026'})
  [Аналитик] Прямой вызов query_db (без bind_tools)

Отчёт:
Рынок AI-агентов в 2025–2026 годах перешёл от экспериментальной стадии к продуктовой: компании и государство всё сильнее фокусируются на прозрачности, безопасности и управлении такими системами. При этом ключевой барьер для массового внедрения — не сама технология, а governance gap: автономные агенты уже умеют действовать в продуктивной среде, но контроль, аудит и юридическая ответственность заметно отстают. Рынок фреймворков фрагментируется по класс

## Часть 2: Агент-как-MCP-сервер (матрёшка)

MCP-сервер `mcp_matryoshka_server.py` (порт 8003) содержит **целый LangGraph-граф** внутри:
- Снаружи: один инструмент `deep_analysis(query) → report`
- Внутри: исследователь → аналитик (два LLM-вызова)

Для вызывающего агента это просто инструмент. Что внутри — деталь реализации.

In [23]:
# Подключаемся к матрёшке (порт 8003)
# uv run mcp_matryoshka_server.py
matryoshka_client = MultiServerMCPClient(
    {
        "analysis": {"transport": "sse", "url": "http://localhost:8003/sse"},
    }
)
matryoshka_tools = await matryoshka_client.get_tools()
print(f"Инструменты матрёшки: {[t.name for t in matryoshka_tools]}")
print(f"Описание: {matryoshka_tools[0].description}")

print("\nВызываем deep_analysis (внутри — 2 агента + 2 LLM-вызова):")
result = await matryoshka_tools[0].ainvoke({"query": "Тренды AI-агентов в 2026 году"})
print(f"\nРезультат:\n{mcp_text(result)}")

Инструменты матрёшки: ['deep_analysis']
Описание: Глубокий анализ темы. Внутри — мультиагентная система (исследователь + аналитик).

    Args:
        query: Тема для анализа
    

Вызываем deep_analysis (внутри — 2 агента + 2 LLM-вызова):

Результат:
Краткий аналитический вывод:

В 2026 году AI-агенты эволюционируют из инструментов общения в **операционных исполнителей**, способных автономно выполнять бизнес-задачи в связке с корпоративными системами. На первый план выйдут **мультимодальность, агентные workflow и глубокая интеграция в процессы компании**, но их массовое внедрение будет сдерживаться требованиями к **контролю, безопасности и аудиту действий**.  

Итог: рынок смещается от “умных ассистентов” к **управляемой автоматизации с агентами**, где ценность будет определяться не самим ИИ, а его встроенностью в реальные бизнес-процессы.


## Часть 3: A2A Agent Card

A2A-сервер (порт 5002) уже запущен в терминале.

Agent Card публикуется по стандартному пути `GET /.well-known/agent.json`.

**Что такое `/.well-known/`?** Это стандарт RFC 8615 — зарезервированный путь на любом HTTP-сервере для метаданных о сервисе. Примеры из реальной жизни:

- `/.well-known/robots.txt` — правила для поисковых ботов
- `/.well-known/openid-configuration` — конфигурация OAuth/OIDC
- `/.well-known/apple-app-site-association` — deep links для iOS

A2A выбрал `/.well-known/agent.json` по тому же принципу: любой клиент знает, где искать описание агента, без предварительной договорённости. Пришёл на сервер → сходил в `/.well-known/agent.json` → узнал что агент умеет, какие навыки, какой формат ввода/вывода.

Как табличка на двери офиса — стандартное место, куда все смотрят.

In [25]:
# Получаем Agent Card по HTTP — стандартный путь A2A v1.0
async with httpx.AsyncClient() as http:
    resp = await http.get("http://127.0.0.1:5002/.well-known/agent.json")
    card = resp.json()

print("=== Agent Card ===")
print(json.dumps(card, indent=2, ensure_ascii=False))

# Поиск агента по навыкам (как в реальном A2A discovery)
print("\n=== Поиск по тегам {legal, compliance} ===")
desired_tags = {"legal", "compliance"}
for skill in card["skills"]:
    if desired_tags & set(skill.get("tags", [])):
        print(f"  Навык: {skill['name']}")
        print(f"  Описание: {skill['description']}")
        print(f"  Примеры: {skill['examples']}")

=== Agent Card ===
{
  "name": "Legal Review Agent",
  "description": "Reviews documents for legal compliance and risks",
  "url": "http://localhost:5002",
  "version": "1.0",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false,
    "stateTransitionHistory": true
  },
  "skills": [
    {
      "id": "legal-review",
      "name": "Legal Compliance Review",
      "description": "Analyzes text for legal risks, compliance issues, and regulatory concerns",
      "tags": [
        "legal",
        "compliance",
        "review"
      ],
      "examples": [
        "Review this contract for potential risks",
        "Check compliance of this policy document"
      ]
    }
  ],
  "securitySchemes": {
    "apiKey": {
      "type": "apiKey",
      "in": "header",
      "name": "X-API-Key"
    }
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain",
    "application/json"
  ]
}

=== Поиск по тегам {legal, compliance} ===
  Навык: Leg

## Часть 4: A2A Task Lifecycle

Отправляем задачи на A2A-сервер через **реальный JSON-RPC 2.0** over HTTP.
Наблюдаем переходы состояний: submitted → working → completed (и input-required).

In [ ]:
# Сценарий 1: полный запрос → completed
print("=== Сценарий 1: полный запрос ===")
async with httpx.AsyncClient() as http:
    resp = await http.post(
        "http://127.0.0.1:5002/",
        json={
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tasks/send",
            "params": {
                "id": "task-001",
                "message": {
                    "role": "user",
                    "parts": [
                        {
                            "kind": "text",
                            "text": (
                                "Проанализируй юридические риски использования AI-агентов "
                                "для автоматического принятия решений в финансовой сфере"
                            ),
                        }
                    ],
                },
            },
        },
        timeout=60,
    )
    result = resp.json()["result"]

print(f"Статус: {result['status']['state']}")
print(f"История переходов: {result['history']}")
print("\nАртефакт (рецензия):")
print(result["artifacts"][0]["parts"][0]["text"][:500])

=== Сценарий 1: полный запрос ===
Статус: completed
История переходов: [{'from': 'submitted', 'to': 'working'}, {'from': 'working', 'to': 'completed'}]

Артефакт (рецензия):
Ниже — краткий юридический разбор рисков использования AI-агентов для автоматического принятия решений в финансовой сфере.

## 1) Основные правовые риски

### 1. Автоматизированное принятие решений в отношении клиентов
Если AI-агент принимает решения по кредитованию, скорингу, инвестиционным рекомендациям, AML/фрод-фильтрации или закрытию счета без участия человека, возникают риски:
- нарушения правил об автоматизированной обработке данных;
- отсутствия права клиента на оспаривание решения;
- не


In [ ]:
# Сценарий 2: слишком короткий запрос → input-required
print("=== Сценарий 2: input-required ===")
async with httpx.AsyncClient() as http:
    resp = await http.post(
        "http://127.0.0.1:5002/",
        json={
            "jsonrpc": "2.0",
            "id": 2,
            "method": "tasks/send",
            "params": {
                "id": "task-002",
                "message": {
                    "role": "user",
                    "parts": [{"kind": "text", "text": "Проверь контракт"}],
                },
            },
        },
    )
    result = resp.json()["result"]

print(f"Статус: {result['status']['state']}")
print(f"Сообщение: {result['status'].get('message', '')}")
print(f"История: {result['history']}")


=== Сценарий 2: input-required ===
Статус: input-required
Сообщение: Пожалуйста, предоставьте более подробное описание для рецензии (минимум 20 символов).
История: [{'from': 'submitted', 'to': 'working'}, {'from': 'working', 'to': 'input-required'}]


In [ ]:
# Проверяем статус через tasks/get (polling)
print("\n--- Проверка через tasks/get (polling) ---")
async with httpx.AsyncClient() as http:
    resp = await http.post(
        "http://127.0.0.1:5002/",
        json={
            "jsonrpc": "2.0",
            "id": 3,
            "method": "tasks/get",
            "params": {"id": "task-002"},
        },
    )
    task = resp.json()["result"]
    print(f"Статус задачи task-002: {task['status']['state']}")


--- Проверка через tasks/get (polling) ---
Статус задачи task-002: input-required


## Часть 4b: A2A через официальный SDK (a2a-sdk)

Выше мы работали с A2A «вручную» — собирали JSON-RPC запросы руками.
Google выпустил официальный Python SDK — **`a2a-sdk`** — который берёт на себя:

- **Discovery**: `ClientFactory.connect(url)` сам идёт в `/.well-known/agent-card.json`
- **Типизация**: `Message`, `TextPart`, `Role` — Pydantic-модели вместо голых dict
- **Транспорт**: HTTP, SSE-стриминг — одним методом `send_message()`

Сравните объём кода ниже с ручным JSON-RPC выше.

In [34]:
# pip install a2a-sdk
import httpx
from a2a.client.client_factory import ClientConfig, ClientFactory
from a2a.types import Message, Role, TextPart

# 1. Discovery — SDK сам скачивает Agent Card
config = ClientConfig(
    httpx_client=httpx.AsyncClient(timeout=60),
    streaming=False,
)
client = await ClientFactory.connect("http://127.0.0.1:5002", client_config=config)
card = await client.get_card()
print(f"Агент: {card.name}")
print(f"Навыки: {[s.name for s in card.skills]}")

# 2. Отправляем задачу — типизированные объекты вместо dict
msg = Message(
    role=Role.user,
    parts=[
        TextPart(
            text=(
                "Проанализируй юридические риски использования AI-агентов "
                "в финансовой сфере"
            )
        )
    ],
    message_id="sdk-msg-001",
)

print("\nОтправляем через SDK...")
async for event in client.send_message(msg):
    if isinstance(event, tuple):
        task, _update = event
        print(f"Статус: {task.status.state}")
        if task.artifacts:
            for artifact in task.artifacts:
                for part in artifact.parts:
                    print(f"\nРецензия:\n{part.root.text[:500]}")

Агент: Legal Review Agent
Навыки: ['Legal Compliance Review']

Отправляем через SDK...
Статус: TaskState.completed

Рецензия:
Ниже — краткий юридический анализ рисков использования AI-агентов в финансовой сфере.

## 1. Регуляторные риски
- **Лицензирование и разрешённая деятельность**: AI-агент может фактически выполнять функции, требующие лицензии или специального статуса (консультирование, управление активами, брокеридж, кредитование, коллекторская деятельность).
- **Нарушение правил disclosure**: недостаточное раскрытие клиенту, что решение/рекомендация сформированы с участием AI.
- **Автоматизированное принятие реш


## Часть 4c: A2A-агент как LangChain Runnable (a2a-langchain-adapters)

`a2a-langchain-adapters` оборачивает **любой** A2A-сервер в LangChain `Runnable`.
Это мост между экосистемами: A2A-агент снаружи → LangChain-узел внутри.

- `A2ARunnable.from_agent_url(url)` — сам скачивает Agent Card, настраивает клиент
- `await agent.ainvoke("текст")` — возвращает `A2AResult` с полями `.text`, `.task_id`, `.status`
- `agent.as_tool(name, desc)` — превращает A2A-агента в LangChain `BaseTool` для `bind_tools`

Покажем оба варианта: прямой вызов как Runnable и как узел LangGraph-графа.

In [ ]:
# pip install a2a-langchain-adapters
from a2a_langchain_adapters import A2ARunnable

# Discovery — скачивает Agent Card, настраивает клиент
agent = await A2ARunnable.from_agent_url("http://127.0.0.1:5002")
print(f"Агент: {agent.agent_card.name}")
print(f"Навыки: {[s.name for s in agent.agent_card.skills]}")

# Прямой вызов как LangChain Runnable
result = await agent.ainvoke(
    "Проанализируй юридические риски использования AI-агентов в HR-процессах"
)
print(f"\nСтатус: {result.status}")
print(f"Task ID: {result.task_id}")
print(f"\nРецензия:\n{result.text[:500]}")

In [ ]:
# A2A-агент как узел LangGraph-графа — всего 3 строки для интеграции

class ReviewState(TypedDict):
    topic: str
    legal_review: str
    summary: str


async def a2a_review_node(state: ReviewState) -> dict:
    """A2A-агент как обычная нода графа — одна строка вызова."""
    result = await agent.ainvoke(f"Юридическая рецензия: {state['topic']}")
    return {"legal_review": result.text or "Рецензия не получена"}


def summary_node(state: ReviewState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Кратко резюмируй юридическую рецензию (2-3 предложения)."),
        HumanMessage(content=state["legal_review"]),
    ])
    return {"summary": response.content}


review_graph = StateGraph(ReviewState)
review_graph.add_node("legal_review", a2a_review_node)   # A2A Runnable
review_graph.add_node("summary", summary_node)            # Обычный LLM
review_graph.add_edge(START, "legal_review")
review_graph.add_edge("legal_review", "summary")
review_graph.add_edge("summary", END)
review_app = review_graph.compile()

print("=== LangGraph + A2A Runnable ===")
result = await review_app.ainvoke({
    "topic": "Автоматическое увольнение сотрудников на основе AI-скоринга",
    "legal_review": "",
    "summary": "",
})
print(f"Резюме:\n{result['summary']}")

# Не забываем закрыть клиент
await agent.close()

## Часть 5: Комбинация — shared state + MCP + A2A

Три уровня коммуникации в одном графе:
- **Shared state** — внутри графа (агенты одной команды)
- **MCP** — для инструментов (DuckDuckGo поиск, SQLite)
- **A2A** — для внешнего агента (юридическая рецензия через HTTP JSON-RPC)

In [ ]:
class CombinedState(TypedDict):
    query: str
    internal_research: str
    external_review: str
    final_report: str


llm = get_llm()


async def internal_researcher(state: CombinedState) -> dict:
    """Внутренний агент: MCP-инструменты → shared state."""
    facts = mcp_text(
        await tools_by_name["search_web"].ainvoke({"query": state["query"]})
    )
    db_data = mcp_text(
        await tools_by_name["query_db"].ainvoke(
            {"sql": "SELECT protocol, version, status FROM protocol_adoption"}
        )
    )
    response = llm.invoke(
        [
            SystemMessage(
                content="Объедини факты из поиска и данные из БД в краткий анализ."
            ),
            HumanMessage(
                content=f"Поиск:\n{facts}\n\nБД:\n{db_data}",
            ),
        ]
    )
    print("  [Внутренний] Исследование через MCP → shared state")
    return {"internal_research": response.content}


async def external_review_node(state: CombinedState) -> dict:
    """Внешний агент: A2A JSON-RPC over HTTP."""
    async with httpx.AsyncClient() as http:
        resp = await http.post(
            "http://127.0.0.1:5002/",
            json={
                "jsonrpc": "2.0",
                "id": 10,
                "method": "tasks/send",
                "params": {
                    "id": "combined-review",
                    "message": {
                        "role": "user",
                        "parts": [
                            {
                                "kind": "text",
                                "text": f"Юридическая рецензия анализа: {state['internal_research'][:300]}",
                            }
                        ],
                    },
                },
            },
            timeout=60,
        )
        result = resp.json()["result"]
    review = (
        result["artifacts"][0]["parts"][0]["text"]
        if result["artifacts"]
        else "Рецензия не получена"
    )
    print(f"  [Внешний A2A] Статус: {result['status']['state']}")
    return {"external_review": review}


def report_writer(state: CombinedState) -> dict:
    """Писатель: объединяет всё через shared state."""
    response = llm.invoke(
        [
            SystemMessage(
                content="Напиши финальный отчёт (3-5 предложений), учитывая анализ и юридическую рецензию."
            ),
            HumanMessage(
                content=f"Анализ:\n{state['internal_research']}\n\nРецензия:\n{state['external_review']}"
            ),
        ]
    )
    print("  [Писатель] Финальный отчёт через shared state")
    return {"final_report": response.content}


graph = StateGraph(CombinedState)
graph.add_node("researcher", internal_researcher)
graph.add_node("external_review", external_review_node)
graph.add_node("writer", report_writer)
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "external_review")
graph.add_edge("external_review", "writer")
graph.add_edge("writer", END)
combined_app = graph.compile()

print("=== Комбинированная система: shared state + MCP + A2A ===")
result = await combined_app.ainvoke(
    {
        "query": "AI agent protocols MCP A2A",
        "internal_research": "",
        "external_review": "",
        "final_report": "",
    }
)

print(f"\nФинальный отчёт:\n{result['final_report']}")

In [ ]:
# Серверы продолжают работать в терминалах.
# Остановите их вручную (Ctrl+C) когда закончите.
print("Готово! Не забудьте остановить серверы в терминалах (Ctrl+C).")